In [1]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "Ntk_silver_ListandLibrary"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Bronze/Lists_Libraries_Combined" # ← Change source path
TARGET_PATH = "abfss://silver/Dim_ListandLibrary" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_silver_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_silver_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_silver_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, eacfab61-6bca-4864-9879-75b649fb2573, 3, Finished, Available, Finished)

🔧 Initializing Ntk_silver_ListandLibrary...
🚀 Starting Ntk_silver_ListandLibrary


In [2]:
source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Bronze_lakehouse.Lakehouse/Files/Bronze_layer/SharePointFiles"
target_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting"

StatementMeta(, eacfab61-6bca-4864-9879-75b649fb2573, 4, Finished, Available, Finished)

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

# Initialize Spark Session
spark = SparkSession.builder.appName("BronzeToSilver_ListsLibrary").getOrCreate()

today = datetime.now()  
from datetime import datetime, timedelta
today = today - timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d") 
bronze_path = f"{source_path}/{year}/{month}/{day}/Lists_Libraries_Combined.csv"

current_date = datetime.now()
year = str(current_date.year)
month = f"{current_date.month:02d}"
day = f"{current_date.day:02d}"
silver_path = f"{target_path}/{year}/{month}/{day}/Dim_ListandLibrary.parquet"

# Load CSV with schema inference
df_raw = spark.read.option("header", "true").option("inferSchema", "true").option("multiline", "true").csv(bronze_path)
df_raw.show(1)

StatementMeta(, eacfab61-6bca-4864-9879-75b649fb2573, 5, Finished, Available, Finished)

+--------------------+-------------+------------------+--------------------+-----------+--------------------+--------------------+--------------------+--------------------+-----------+------------+------------+---------+-----------+---------+----------------+----------+--------------------+--------------------+-----+----------+---------+--------------+--------------------+-------------------+------+---------+-----------------+---------+-------------------+-----------------+--------------------+----------------+-----------------+----------------+------------+----------------+----------------+--------------+------------------+------------------+---------------+-----------------+--------------------+--------------+------------------+-------------------+-------------+----------------+----+-------------------+-----------------+----------------------+-------------+--------------------------------+
|             SiteUrl|     SiteName|             Title|                  Id|Description|       

In [4]:
# Null Analysis
null_counts = df_raw.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_raw.columns])

# Whitespace trimming for string columns
string_cols = [field.name for field in df_raw.schema.fields if field.dataType == StringType()]
df_trimmed = df_raw
for col_name in string_cols:
    df_trimmed = df_trimmed.withColumn(col_name, trim(col(col_name)))

# Null standardization
df_null_std = df_trimmed
for col_name in string_cols:
    df_null_std = df_null_std.withColumn(
        col_name,
        when((col(col_name) == "") | (upper(col(col_name)) == "NULL") | (upper(col(col_name)) == "N/A"), None)
        .otherwise(col(col_name))
    )

# Boolean normalization - identify boolean columns dynamically
df_bool_norm = df_null_std
for col_name in df_null_std.columns:
    sample_vals = df_null_std.select(col_name).distinct().rdd.flatMap(lambda x: x).collect()
    bool_vals = {'TRUE', 'FALSE', 'True', 'False', 'true', 'false', '1', '0'}
    if any(str(val).upper() in {'TRUE', 'FALSE'} for val in sample_vals if val is not None):
        df_bool_norm = df_bool_norm.withColumn(
            col_name,
            when(upper(trim(col(col_name))).isin(["TRUE", "1"]), True)
            .when(upper(trim(col(col_name))).isin(["FALSE", "0"]), False)
            .otherwise(None)
        )

# Numeric normalization - detect numeric columns
df_numeric = df_bool_norm
for col_name in df_bool_norm.columns:
    if col_name in string_cols:
        sample_val = df_bool_norm.select(col_name).filter(col(col_name).isNotNull()).first()
        if sample_val and sample_val[0] and str(sample_val[0]).replace(',','').isdigit():
            df_numeric = df_numeric.withColumn(
                col_name,
                when(col(col_name).rlike("^[0-9,]+$"), regexp_replace(col(col_name), ",", "").cast(IntegerType()))
                .otherwise(None)
            )

# Date normalization - detect date columns
# Date normalization - detect date columns
df_dates = df_numeric
for col_name in df_numeric.columns:
    if 'date' in col_name.lower() or 'created' in col_name.lower() or 'modified' in col_name.lower():
        df_dates = df_dates.withColumn(
            col_name,
            to_timestamp(col(col_name), "M/d/yyyy h:mm:ss a")
        )



StatementMeta(, eacfab61-6bca-4864-9879-75b649fb2573, 6, Finished, Available, Finished)

In [5]:

# Metadata enrichment
df_enriched = df_dates.withColumn("SnapshotDate", current_timestamp()) \
    .withColumn("data_source", lit("SharePoint_Lists_Libraries")) \
    .withColumn("batch_id", lit(f"{year}{month}{day}"))

# Add calculated columns if URL columns exist
url_cols = [c for c in df_enriched.columns if 'url' in c.lower()]
if url_cols:
    df_enriched = df_enriched.withColumn("site_domain", regexp_extract(col(url_cols[0]), "https://([^/]+)", 1))

# Add item count analysis if ItemCount exists
if 'ItemCount' in df_enriched.columns:
    df_enriched = df_enriched.withColumn(
        "usage_category",
        when(col("ItemCount") == 0, "Empty")
        .when(col("ItemCount").between(1, 10), "Low")
        .when(col("ItemCount").between(11, 100), "Medium")
        .when(col("ItemCount") > 100, "High")
        .otherwise("Unknown")
    )

# Column renaming to snake_case
df_renamed = df_enriched
for col_name in df_enriched.columns:
    new_name = ''.join(['_' + c.lower() if c.isupper() and i > 0 else c.lower() for i, c in enumerate(col_name)])
    if new_name != col_name:
        df_renamed = df_renamed.withColumnRenamed(col_name, new_name)

# Deduplication - use first column as key
key_col = df_renamed.columns[0]
if 'id' in key_col.lower():
    df_deduped = df_renamed.dropDuplicates([key_col])
else:
    df_deduped = df_renamed.dropDuplicates()

# Filtering - remove records where key column is null
df_filtered = df_deduped.filter(col(key_col).isNotNull())

# Hierarchy detection for URL-based data
if any('url' in c.lower() for c in df_filtered.columns):
    site_col = [c for c in df_filtered.columns if 'site' in c.lower() and 'url' in c.lower()][0] if any('site' in c.lower() and 'url' in c.lower() for c in df_filtered.columns) else None
    if site_col:
        df_filtered = df_filtered.withColumn("hierarchy_level", size(split(regexp_extract(col(site_col), "https://[^/]+(/.*)", 1), "/")))

# Grouping & Aggregation
numeric_cols = [field.name for field in df_filtered.schema.fields if field.dataType in [IntegerType(), LongType(), DoubleType(), FloatType()]]
if len(numeric_cols) > 0:
    agg_expressions = [count("*").alias("record_count")]
    for num_col in numeric_cols[:3]:  # Limit to first 3 numeric columns
        agg_expressions.extend([
            sum(num_col).alias(f"total_{num_col}"),
            avg(num_col).alias(f"avg_{num_col}")
        ])
    
    summary_stats = df_filtered.agg(*agg_expressions)


StatementMeta(, eacfab61-6bca-4864-9879-75b649fb2573, 7, Finished, Available, Finished)

In [6]:

# Distribution Analysis
categorical_cols = [field.name for field in df_filtered.schema.fields if field.dataType == StringType()][:5]
distributions = {}
for cat_col in categorical_cols:
    if df_filtered.select(cat_col).distinct().count() < 50:
        distributions[cat_col] = df_filtered.groupBy(cat_col).count().orderBy(desc("count"))

# Distinct counts
distinct_counts = df_filtered.select([approx_count_distinct(c).alias(f"distinct_{c}") for c in df_filtered.columns[:10]])


# Data quality scoring
from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

quality_exprs = [when(col(c).isNotNull(), 1).otherwise(0) for c in df_filtered.columns[:5]]
df_final = df_filtered.withColumn("data_quality_score", reduce(lambda x, y: x + y, quality_exprs))

df_final = df_final.withColumnRenamed("id", "Itemid").withColumnRenamed("url", "Itemurl")

print(df_final.count())
df_final.printSchema()


StatementMeta(, eacfab61-6bca-4864-9879-75b649fb2573, 8, Finished, Available, Finished)

3
root
 |-- site_url: string (nullable = true)
 |-- site_name: string (nullable = true)
 |-- title: string (nullable = true)
 |-- Itemid: string (nullable = true)
 |-- description: string (nullable = true)
 |-- Itemurl: string (nullable = true)
 |-- default_view_url: string (nullable = true)
 |-- root_folder_url: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- base_type: string (nullable = true)
 |-- base_template: integer (nullable = true)
 |-- list_template: string (nullable = true)
 |-- item_count: integer (nullable = true)
 |-- folder_count: integer (nullable = true)
 |-- view_count: integer (nullable = true)
 |-- content_type_count: integer (nullable = true)
 |-- field_count: integer (nullable = true)
 |-- created: timestamp (nullable = true)
 |-- last_modified: timestamp (nullable = true)
 |-- owner: string (nullable = true)
 |-- owner_email: string (nullable = true)
 |-- created_by: timestamp (nullable = true)
 |-- created_by_email: timestamp (nullable = t

In [7]:
# Save to Silver layer
# df_final.repartition(2).write.mode("overwrite").option("compression", "snappy").parquet(silver_path)
# Creating key using hash value for siteurl
df_final = df_final.withColumn(
    "ObjectKey",
    when(
        col("site_url").isNotNull(),
        sha2(col("site_url"), 256)
    ).otherwise(None)
)

df_final.write.format("parquet").mode("overwrite").save(silver_path)

print("Dim_ListandLibrary Parquet file created successfully in Silver layer!")
print(df_final.count())
df_final.printSchema()

StatementMeta(, eacfab61-6bca-4864-9879-75b649fb2573, 9, Finished, Available, Finished)

Dim_ListandLibrary Parquet file created successfully in Silver layer!
3
root
 |-- site_url: string (nullable = true)
 |-- site_name: string (nullable = true)
 |-- title: string (nullable = true)
 |-- Itemid: string (nullable = true)
 |-- description: string (nullable = true)
 |-- Itemurl: string (nullable = true)
 |-- default_view_url: string (nullable = true)
 |-- root_folder_url: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- base_type: string (nullable = true)
 |-- base_template: integer (nullable = true)
 |-- list_template: string (nullable = true)
 |-- item_count: integer (nullable = true)
 |-- folder_count: integer (nullable = true)
 |-- view_count: integer (nullable = true)
 |-- content_type_count: integer (nullable = true)
 |-- field_count: integer (nullable = true)
 |-- created: timestamp (nullable = true)
 |-- last_modified: timestamp (nullable = true)
 |-- owner: string (nullable = true)
 |-- owner_email: string (nullable = true)
 |-- created_by: time

In [8]:
from datetime import datetime

# Define the log_etl_activity function for logging ETL process
def log_etl_activity(status, start_time, rows_read=None, rows_written=None, bytes_processed=None, error_details=None):

    end_time = datetime.now()
    duration_seconds = (end_time - start_time).total_seconds()

    log_message = {
        'Status': status,
        'StartTime': start_time,
        'EndTime': end_time,
        'DurationSeconds': duration_seconds,
        'RowsRead': rows_read,
        'RowsWritten': rows_written,
        'BytesProcessed': bytes_processed,
        'ErrorDetails': error_details
    }

    # For simplicity, let's print the log message (this can be replaced with a logging system)
    print("Logging ETL Activity:", log_message)

# Ensure processing_successful is defined before this block
try:
    NOTEBOOK_NAME = "ETL_Pipeline_Example"  # Define your notebook name or use the existing one
    start_time = datetime.now()  # Capture the start time of the ETL process

    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Simulated metrics
    rows_read = df_raw.count()   # Correct this to have a meaningful `rows_read`
    rows_written = df_final.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details=error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, eacfab61-6bca-4864-9879-75b649fb2573, 10, Finished, Available, Finished)

🔄 Starting ETL processing for ETL_Pipeline_Example...
Logging ETL Activity: {'Status': 'SUCCESS', 'StartTime': datetime.datetime(2025, 10, 15, 5, 24, 37, 34914), 'EndTime': datetime.datetime(2025, 10, 15, 5, 24, 37, 894710), 'DurationSeconds': 0.859796, 'RowsRead': 3, 'RowsWritten': 3, 'BytesProcessed': 524288000, 'ErrorDetails': None}
🎉 ETL_Pipeline_Example pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 3 → 3
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ETL_Pipeline_Example:
+-----+------+---------+-------+---------------+--------+-----------+
|LogID|Status|StartTime|EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+------+---------+-------+---------------+--------+-----------+
+-----+------+---------+-------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ETL_Pipeline_Example logging completed!
